# Paradas: símbolo, nome e dica — Mobilidade Urbana — v11
- **Web map**: paradas com símbolo novo (círculo branco, borda azul GeoAPUC #4463a9, ônibus azul), desenhadas por cima das linhas.
- **Widget Horários da Linha v7**: nome da parada ao passar o mouse (ou tocar) + dica "💡 Ligue Mostrar paradas…"
  (aparece com o interruptor desligado; clicar na dica já liga).
- O config do app **não muda**.

⚠ **Widget mudou → reinicie o client** (`cd C:\ExB_1.19\client` → `npm start`, ou o Iniciar_ExB.bat) →
construtor **Ctrl+F5 → mudança pequena → Salvar → Publicar** → teste na prévia (https).
Saída: `Documents\Saidas_Notebooks\Paradas_Simbolo_Dica_v11.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, shutil
NB_NOME = "Paradas_Simbolo_Dica_v11"
WIDGETS_DIR = r"C:\ExB_1.19\client\your-extensions\widgets"
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_paradas_v11_{datetime.datetime.now():%Y%m%d_%H%M%S}"); os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Widget Horários da Linha v7 =====
log("\n▶ CÉLULA 2 — Atualização do widget: iniciando...")
ARQUIVOS_WIDGET = {
    'manifest.json': r'''{
  "name": "horarios-linha",
  "label": "Horários da Linha",
  "type": "widget",
  "version": "1.19.0",
  "exbVersion": "1.19.0",
  "author": "GeoAPUC",
  "description": "Mostra a tabela de horários (dias úteis) da linha de ônibus selecionada na lista, com a próxima saída em destaque.",
  "copyright": "",
  "license": "",
  "properties": {},
  "translatedLocales": [
    "en"
  ],
  "defaultSize": {
    "width": 380,
    "height": 560
  },
  "dependency": [
    "jimu-arcgis"
  ]
}''',
    'config.json': r'''{
  "dataSourceId": "dataSource_1-linhas_onibus",
  "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
  "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
  "tema": "azul",
  "mobilibusUrl": "https://editor.mobilibus.com/web/detalhes-linha/4abp9",
  "mobilibusLinhas": {
    "110": "fpt",
    "115": "2cjn",
    "120": "fqg",
    "125": "fqa",
    "130": "fqp",
    "135": "fq7",
    "140": "fq3",
    "145": "fq4",
    "150": "fpo",
    "155": "fr0",
    "160": "fqt",
    "165": "fqo",
    "170": "fr1",
    "175": "6ckh",
    "195": "fpg",
    "200": "fpm",
    "205": "fps",
    "210": "fqr",
    "215": "hrte",
    "220": "fpr",
    "230": "fq8",
    "235": "fqk",
    "240": "fph",
    "270": "9t55",
    "320": "fpu",
    "400": "fqn",
    "500": "fq6",
    "998": "fqd",
    "999": "fqb"
  },
  "paradasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Paradas_Onibus/FeatureServer/0",
  "paradasLayerId": "paradas_onibus",
  "paradasDistanciaM": 30,
  "paradasLigadoAoAbrir": false,
  "paradasModo": "campo"
}''',
    'icon.svg': r'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 16 16" width="16" height="16"><rect x="3" y="1.5" width="10" height="11" rx="2" fill="none" stroke="#000" stroke-width="1.2"/><rect x="4.6" y="3.2" width="6.8" height="3.6" rx=".6" fill="#000"/><circle cx="5.5" cy="9.6" r=".9" fill="#000"/><circle cx="10.5" cy="9.6" r=".9" fill="#000"/><rect x="4" y="12.5" width="2" height="2" rx=".5" fill="#000"/><rect x="10" y="12.5" width="2" height="2" rx=".5" fill="#000"/></svg>
''',
    'src/runtime/widget.tsx': r'''// Horários da Linha – GeoAPUC (v7: dica do 'Mostrar paradas' + nome da parada ao passar o mouse/tocar)
// Mostra os horários (dias úteis) da linha selecionada na lista de linhas.
// Escrito sem JSX (React.createElement) para não depender da configuração de compilação.
import { React, getAppStore, type AllWidgetProps } from 'jimu-core'
import { JimuMapViewComponent } from 'jimu-arcgis'

const h = React.createElement
const { useState, useEffect } = React

const TEMAS: any = {
  escuro: { fundo: '#0f1e33', card: '#16294a', borda: 'rgba(255,255,255,0.10)', texto: '#ffffff', suave: '#9fb0cc',
            destaque: '#F5A623', azul: '#4285F4', verde: '#1D9E75', chip: '#1f3b66', aviso: 'rgba(245,166,35,0.14)' },
  azul: { fundo: '#4463a9', card: 'rgba(255,255,255,0.13)', borda: 'rgba(255,255,255,0.25)', texto: '#ffffff', suave: '#dbe4f5',
          destaque: '#F5A623', azul: '#1b2a4a', verde: '#1D9E75', chip: 'rgba(255,255,255,0.20)', aviso: 'rgba(245,166,35,0.22)' },
  claro: { fundo: '#ffffff', card: '#f4f6fb', borda: '#dfe5ef', texto: '#1b2a4a', suave: '#6b7690',
           destaque: '#4463a9', azul: '#4463a9', verde: '#1f9d55', chip: '#e6ecf8', aviso: '#FFF4E0' }
}

function hhmm (v: any): string {
  const s = String(v ?? '').trim()
  const m = s.match(/^(\d{1,2}):(\d{2})/)
  return m ? `${m[1].padStart(2, '0')}:${m[2]}` : s
}
function minutos (v: string): number {
  const m = v.match(/^(\d{2}):(\d{2})/)
  return m ? Number(m[1]) * 60 + Number(m[2]) : -1
}
function chaveSentido (v: any): string {
  return String(v ?? '').normalize('NFD').replace(/[\u0300-\u036f]/g, '').toLowerCase()
    .replace(/sentido para/g, '').replace(/[^a-z0-9]/g, '')
}
async function contar (url: string, where: string): Promise<number> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&returnCountOnly=true&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na contagem')
  return Number(j.count || 0)
}
function aspas (v: any): string { return String(v ?? '').replace(/'/g, "''") }

async function consultar (url: string, where: string, campos: string): Promise<any[]> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&outFields=${campos}&returnGeometry=false&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na consulta')
  return (j.features || []).map((f: any) => f.attributes || {})
}

async function geometriaLinha (urlLin: string, oid: string): Promise<any> {
  const q = `${urlLin}/query?where=${encodeURIComponent('objectid=' + Number(oid))}&returnGeometry=true&outFields=objectid&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na geometria')
  const f = (j.features || [])[0]
  if (!f?.geometry) throw new Error('linha sem geometria')
  const g = f.geometry
  if (!g.spatialReference) g.spatialReference = j.spatialReference
  return g
}
async function paradasPerto (urlPar: string, geom: any, metros: number): Promise<number[]> {
  const corpo = new URLSearchParams({
    geometry: JSON.stringify(geom), geometryType: 'esriGeometryPolyline', spatialRel: 'esriSpatialRelIntersects',
    distance: String(metros), units: 'esriSRUnit_Meter', returnIdsOnly: 'true', f: 'json'
  })
  const j: any = await fetch(`${urlPar}/query`, { method: 'POST', body: corpo }).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro nas paradas')
  return j.objectIds || []
}

function relogio (cor: string, tam: number) {
  return h('svg', { width: tam, height: tam, viewBox: '0 0 24 24', style: { flex: 'none' } },
    h('circle', { cx: 12, cy: 12, r: 10, fill: '#ffffff', stroke: cor, strokeWidth: 2 }),
    h('path', { d: 'M12 6v6l4 2', fill: 'none', stroke: '#c0392b', strokeWidth: 2, strokeLinecap: 'round' }))
}

export default function Widget (props: AllWidgetProps<any>) {
  const cfg: any = props.config || {}
  const dsId: string = cfg.dataSourceId || 'dataSource_1-linhas_onibus'
  const urlLin: string = cfg.linhasUrl
  const urlHor: string = cfg.horariosUrl
  const T: any = TEMAS[cfg.tema] || TEMAS.escuro
  const mbBase: string = cfg.mobilibusUrl || 'https://editor.mobilibus.com/web/detalhes-linha/4abp9'
  const mbCod: any = cfg.mobilibusLinhas || {}
  const linkTempoReal = (cod: any): string => {
    const c = mbCod[String(cod ?? '').trim()]
    return c ? `${mbBase}#${c}` : mbBase
  }
  const botaoTempoReal = (cod: any, texto: string) => h('a', {
    href: linkTempoReal(cod), target: '_blank', rel: 'noopener noreferrer',
    style: { display: 'flex', alignItems: 'center', justifyContent: 'center', gap: 8, padding: '9px 12px', borderRadius: 999,
             background: '#d64545', color: '#ffffff', fontWeight: 700, fontSize: 13, textDecoration: 'none' }
  }, h('span', { style: { width: 9, height: 9, borderRadius: '50%', background: '#ffffff', display: 'inline-block' } }), texto)

  // id do registro selecionado na lista (lido do estado do app e acompanhado por assinatura)
  const lerSelecao = (): string | null => {
    try {
      const inf: any = (getAppStore().getState() as any)?.dataSourcesInfo || {}
      for (const k of Object.keys(inf)) {
        if (k === dsId || k.startsWith(dsId + '-')) {
          const ids = inf[k]?.selectedIds
          if (ids && ids.length) return String(ids[0])
        }
      }
    } catch (e) { console.warn('[Horários da Linha] seleção', e) }
    return null
  }
  const [oid, setOid] = useState<string | null>(lerSelecao())
  useEffect(() => {
    const store: any = getAppStore()
    const desfazer = store.subscribe(() => {
      const novo = lerSelecao()
      setOid(atual => (atual === novo ? atual : novo))
    })
    return () => { try { desfazer() } catch (e) {} }
  }, [dsId])

  const [linha, setLinha] = useState<any>(null)
  const [hor, setHor] = useState<any[]>([])
  const [planoB, setPlanoB] = useState<boolean>(false)
  const [carregando, setCarregando] = useState<boolean>(false)
  const [erro, setErro] = useState<string | null>(null)
  const [agora, setAgora] = useState<Date>(new Date())

  useEffect(() => { const t = setInterval(() => setAgora(new Date()), 60000); return () => clearInterval(t) }, [])

  // ---- interruptor "Mostrar paradas" ----
  const urlPar: string = cfg.paradasUrl
  const idCamadaParadas: string = cfg.paradasLayerId || 'pontos_onibus'
  const metros: number = Number(cfg.paradasDistanciaM || 30)
  const [jmv, setJmv] = useState<any>(null)
  const [mostrar, setMostrar] = useState<boolean>(!!cfg.paradasLigadoAoAbrir)
  const [nParadas, setNParadas] = useState<number | null>(null)
  useEffect(() => {
    const v: any = jmv?.view
    if (!v || !urlPar) return
    const lyr: any = v.map.allLayers.find((l: any) => l.id === idCamadaParadas)
    if (!lyr) { console.warn('[Horários da Linha] camada de paradas não encontrada:', idCamadaParadas); return }
    let vivo = true
    if (!mostrar) { lyr.visible = false; setNParadas(null); return }
    ;(async () => {
      try {
        await lyr.load()
        const oidCampo = lyr.objectIdField || 'objectid'
        if (!oid) { lyr.definitionExpression = '1=0'; lyr.visible = true; if (vivo) setNParadas(0); return }
        if (cfg.paradasModo === 'campo') {
          // camada de Paradas (Mobilibus): campos 'viagens' (;cod:sentido;) e 'linhas' (,cod,)
          const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'cod_linha,sentido'))[0] || {}
          const cod = String(l.cod_linha ?? '').trim()
          let where = `viagens LIKE '%;${aspas(cod + ':' + chaveSentido(l.sentido))};%'`
          let n = await contar(urlPar, where)
          if (!n) { where = `linhas LIKE '%,${aspas(cod)},%'`; n = await contar(urlPar, where) }
          if (!vivo) return
          lyr.definitionExpression = n ? where : '1=0'
          lyr.visible = true
          setNParadas(n)
        } else {
          const geom = await geometriaLinha(urlLin, oid)
          const ids = await paradasPerto(urlPar, geom, metros)
          if (!vivo) return
          lyr.definitionExpression = ids.length ? `${oidCampo} IN (${ids.join(',')})` : '1=0'
          lyr.visible = true
          setNParadas(ids.length)
        }
      } catch (e) {
        console.warn('[Horários da Linha] paradas', e)
        if (vivo) { setNParadas(null) }
      }
    })()
    return () => { vivo = false }
  }, [jmv, mostrar, oid, urlPar, urlLin, idCamadaParadas, metros])

  // nome da parada ao passar o mouse (ou tocar) sobre ela
  useEffect(() => {
    const v: any = jmv?.view
    if (!v) return
    const lyr: any = v.map.allLayers.find((l: any) => l.id === idCamadaParadas)
    if (!lyr) return
    try { lyr.outFields = ['nome'] } catch (e) {}
    const dica = document.createElement('div')
    Object.assign(dica.style, { position: 'absolute', display: 'none', pointerEvents: 'none', zIndex: '99', maxWidth: '260px',
      background: '#ffffff', color: '#1b2a4a', border: '2px solid #4463a9', borderRadius: '8px', padding: '4px 8px',
      fontSize: '12px', fontWeight: '600', boxShadow: '0 2px 6px rgba(0,0,0,.25)', whiteSpace: 'nowrap' })
    v.container.appendChild(dica)
    let ocupado = false
    const mostrarDica = (ev: any) => {
      if (ocupado || !lyr.visible) { if (!lyr.visible) dica.style.display = 'none'; return }
      ocupado = true
      v.hitTest(ev, { include: [lyr] }).then((res: any) => {
        ocupado = false
        const g = (res?.results || []).find((r: any) => r.graphic?.layer?.id === idCamadaParadas)?.graphic
        if (g) {
          dica.textContent = '🚏 ' + (g.attributes?.nome || 'Parada de ônibus')
          dica.style.left = (ev.x + 14) + 'px'; dica.style.top = (ev.y + 14) + 'px'; dica.style.display = 'block'
          v.container.style.cursor = 'pointer'
        } else { dica.style.display = 'none'; v.container.style.cursor = '' }
      }).catch(() => { ocupado = false })
    }
    const h1 = v.on('pointer-move', mostrarDica)
    const h2 = v.on('click', mostrarDica)
    const h3 = v.on('pointer-leave', () => { dica.style.display = 'none' })
    return () => { try { h1.remove(); h2.remove(); h3.remove(); dica.remove() } catch (e) {} }
  }, [jmv, idCamadaParadas])

  const dicaParadas = (texto: string) => h('div', {
    onClick: () => setMostrar(true), title: 'Clique para ligar',
    style: { cursor: 'pointer', background: 'rgba(255,255,255,0.14)', border: `1px dashed ${T.destaque}`, borderRadius: 8,
             padding: '7px 10px', fontSize: 12, color: T.texto, lineHeight: 1.35 }
  }, '💡 ', texto)

  const interruptor = h('div', {
    onClick: () => setMostrar(!mostrar), role: 'switch', 'aria-checked': mostrar, title: 'Mostrar no mapa só as paradas da linha selecionada',
    style: { display: 'flex', alignItems: 'center', gap: 8, cursor: 'pointer', userSelect: 'none', fontSize: 12, color: T.texto }
  },
    h('span', null, 'Mostrar paradas' + (mostrar && nParadas !== null ? ` (${nParadas})` : '')),
    h('span', { style: { width: 38, height: 20, borderRadius: 999, position: 'relative', flex: 'none',
                         background: mostrar ? T.destaque : 'rgba(255,255,255,0.35)', transition: 'background .2s' } },
      h('span', { style: { position: 'absolute', top: 2, left: mostrar ? 20 : 2, width: 16, height: 16, borderRadius: '50%',
                           background: '#ffffff', boxShadow: '0 1px 2px rgba(0,0,0,.3)', transition: 'left .2s' } })))
  const mapaId: any = (props as any).useMapWidgetIds?.[0]
  const conector = mapaId ? h(JimuMapViewComponent as any, { useMapWidgetId: mapaId, onActiveViewChange: (v: any) => setJmv(v) }) : null

  useEffect(() => {
    let vivo = true
    if (!oid) { setLinha(null); setHor([]); return }
    setCarregando(true); setErro(null)
    ;(async () => {
      try {
        const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'objectid,id_rota,cod_linha,nome,sentido'))[0]
        if (!l) throw new Error('linha não encontrada')
        let rows = await consultar(urlHor, `id_rota='${aspas(l.id_rota)}'`, 'id_rota,ponto,horario,sentido,observacao')
        let b = false
        if (!rows.length) {
          const cod = String(l.id_rota || '').split('_')[0]
          rows = (await consultar(urlHor, `id_rota LIKE '${aspas(cod)}%'`, 'id_rota,ponto,horario,sentido,observacao'))
            .filter((r: any) => String(r.id_rota || '').startsWith(cod + '_'))
          b = rows.length > 0
        }
        rows.forEach((r: any) => { r._h = hhmm(r.horario); r._m = minutos(r._h) })
        rows.sort((a: any, z: any) => a._m - z._m)
        if (vivo) { setLinha(l); setHor(rows); setPlanoB(b); setCarregando(false) }
      } catch (e: any) {
        console.warn('[Horários da Linha]', e)
        if (vivo) { setErro('Não foi possível carregar os horários.'); setCarregando(false) }
      }
    })()
    return () => { vivo = false }
  }, [oid, urlLin, urlHor])

  const chip = (txt: string, bg: string, cor = '#ffffff') =>
    h('span', { style: { display: 'inline-block', padding: '2px 10px', borderRadius: 999, background: bg, color: cor, fontSize: 11, fontWeight: 700 } }, txt)
  const rotulo = (txt: string) => h('div', { style: { fontSize: 10, letterSpacing: 0.6, color: T.suave, textTransform: 'uppercase' } }, txt)

  const caixa = (filhos: any[]) => h('div', {
    className: 'jimu-widget horarios-linha',
    style: { width: '100%', height: '100%', boxSizing: 'border-box', background: T.fundo, color: T.texto, borderRadius: 12,
             padding: 14, overflowY: 'auto', fontFamily: 'inherit', display: 'flex', flexDirection: 'column', gap: 12,
             border: `1px solid ${T.borda}` }
  }, conector,
    h('div', { style: { display: 'flex', alignItems: 'center', justifyContent: 'space-between', gap: 8 } },
      h('div', { style: { fontSize: 15, fontWeight: 600, color: T.texto } }, 'Tabela de Horários'), interruptor),
    ...filhos)

  if (!oid) {
    return caixa([h('div', { style: { margin: 'auto', textAlign: 'center', color: T.suave, fontSize: 13, lineHeight: 1.5 } },
      h('div', { style: { fontSize: 34, marginBottom: 6 } }, '🚌'), 'Selecione uma linha na lista', h('br'), 'para ver os horários.'),
      botaoTempoReal(null, 'Onde está meu ônibus? · tempo real'),
      dicaParadas('Dica: depois de escolher a linha, ligue “Mostrar paradas” para ver os pontos de ônibus dela no mapa.')])
  }
  if (carregando) return caixa([h('div', { style: { margin: 'auto', color: T.suave } }, 'Carregando horários…')])
  if (erro || !linha) return caixa([h('div', { style: { margin: 'auto', color: '#e57373' } }, erro || '—')])

  try {
    // próxima saída (só em dias úteis)
    const dow = agora.getDay()
    const diaUtil = dow >= 1 && dow <= 5
    const minAgora = agora.getHours() * 60 + agora.getMinutes()
    const proxima = diaUtil ? hor.find((r: any) => r._m >= minAgora) : null
    let destaque: string = proxima ? proxima._h : (hor[0]?._h || '—')
    let sub = ''
    if (!hor.length) sub = 'Sem horários na base da Prefeitura — veja no botão de tempo real.'
    else if (!diaUtil) sub = 'Fim de semana — horários válidos para dias úteis. Primeira saída:'
    else if (proxima) { const d = proxima._m - minAgora; sub = d === 0 ? 'Saindo agora' : `Próxima saída · em ${d >= 60 ? Math.floor(d / 60) + ' h ' : ''}${d % 60} min` }
    else sub = 'Não há mais saídas hoje. Primeira saída amanhã:'

    // agrupa por sentido (plano B) e por ponto
    const grupos: any = {}
    hor.forEach((r: any) => {
      const k = (planoB ? (r.sentido || 'Outro sentido') + ' · ' : '') + (r.ponto || '—')
      ;(grupos[k] = grupos[k] || []).push(r)
    })
    const obs = Array.from(new Set(hor.map((r: any) => String(r.observacao || '').trim()).filter((x: string) => x && x !== '-')))

    return caixa([
      // cabeçalho da linha
      h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: 10 } },
        h('div', { style: { minWidth: 46, textAlign: 'center', padding: '8px 6px', borderRadius: 8, border: `1px solid ${T.destaque}`,
                            color: T.destaque, fontWeight: 800, fontSize: 17, background: 'rgba(245,166,35,0.10)' } }, String(linha.cod_linha ?? '')),
        h('div', { style: { display: 'flex', flexDirection: 'column', gap: 4, minWidth: 0 } },
          h('div', { style: { fontSize: 16, fontWeight: 700, lineHeight: 1.2 } }, linha.nome || ''),
          h('div', { style: { display: 'flex', gap: 6, flexWrap: 'wrap' } }, chip('Dias úteis', T.azul), chip(linha.sentido || '', T.verde)))),
      botaoTempoReal(linha.cod_linha, `Onde está o ônibus da linha ${linha.cod_linha}? · tempo real`),
      !mostrar ? dicaParadas(`Ligue “Mostrar paradas” (acima) para ver no mapa as paradas da linha ${linha.cod_linha} neste sentido — passe o mouse sobre uma parada para ver o nome.`) : null,
      // próxima saída
      h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: '10px 12px' } },
        relogio(T.suave, 34),
        h('div', null, rotulo(sub || 'Horário'), h('div', { style: { fontSize: 32, fontWeight: 800, lineHeight: 1.1 } }, destaque))),
      planoB
        ? h('div', { style: { background: T.aviso, borderLeft: `4px solid ${T.destaque}`, borderRadius: 6, padding: '6px 10px', fontSize: 12, color: T.texto } },
            `Este sentido não tem horários próprios. Abaixo, os horários de saída da linha ${linha.cod_linha} nos outros sentidos.`)
        : null,
      // todos os horários
      ...Object.keys(grupos).map((k: string) => h('div', { key: k, style: { display: 'flex', flexDirection: 'column', gap: 6 } },
        rotulo('Ponto · ' + k),
        h('div', { style: { display: 'flex', flexWrap: 'wrap', gap: 6 } },
          ...grupos[k].map((r: any, i: number) => {
            const passou = diaUtil && r._m < minAgora
            const ehProx = proxima && r === proxima
            return h('span', { key: i, style: { padding: '4px 9px', borderRadius: 6, fontWeight: 700, fontSize: 13,
              background: ehProx ? T.destaque : T.chip, color: ehProx ? '#1b2a4a' : T.texto, opacity: passou ? 0.4 : 1 } }, r._h)
          })))),
      obs.length ? h('div', null, rotulo('Observação'), h('div', { style: { fontSize: 12, color: T.suave } }, obs.join(' · '))) : null,
      h('div', { style: { fontSize: 11, color: T.suave, marginTop: 'auto' } }, `${hor.length} horários · fonte: setor de transportes`)
    ])
  } catch (e: any) {
    console.error('[Horários da Linha]', e)
    return caixa([h('div', { style: { color: '#e57373', fontSize: 12 } }, 'Erro no widget: ' + String(e?.message || e))])
  }
}
''',
    'src/runtime/translations/default.ts': r'''export default {
  _widgetLabel: 'Horários da Linha'
}
''',
}

destino = os.path.join(WIDGETS_DIR, "horarios-linha")
if os.path.isdir(destino):
    shutil.copytree(destino, os.path.join(PASTA_BKP, "widget_horarios-linha_antes"), dirs_exist_ok=True)
for rel, conteudo in ARQUIVOS_WIDGET.items():
    p = os.path.join(destino, *rel.split("/"))
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        f.write(conteudo)
    log("  ·", p)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Web map: símbolo novo das paradas =====
log("\n▶ CÉLULA 3 — Símbolo das paradas no web map: iniciando...")
from arcgis.gis import GIS
gis = GIS("pro")
SIMBOLO_PNG = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAYAAACqaXHeAAACg0lEQVR4Xu2a21HDMBBFA0Np0AIUAGVBAdAC6Q1GH2YUxdKefRkl8fm2pb1Xd9cm5nDYuW3utpT/+Pr5Q/Y7frxsVlfqRlSwRKYh4QZEie4RbUaYAdnCW6KMcBuwtfBoI+6l/hvx3+IL3hrMCbBs/P3+jPx+evtC19VYW0JtgFY4Fd1Da4bWCJUBVLxXtNcMjQnYACI+S3gLSQU1AQ3BmcQXiNGk5gIyQIIUFE3UnmILjJyMKsLLqCWkVhga4BVPepUimW01oWuA1EPWgqT7Cj0xo3sls3tJMM0ASYRUjJXRulJNPVYT4Il+lvgaaxLWUvCgOQ2Ly5Z7LM/9hbKf5hDOWkDqfQ0R4ulzn7CmDc+AKDFboKkVG3CtnBgQGf9ZaTWiBGgiNQu0ZmTANbMbsJxur/9plGakV3utdU9A5slp3shGRK2zhupV2EJm8RGkJuAS2A24hFPK5OYNSB+C7Z+zZCjWz29yvYf0BLQvI9KLlfZ6L38G9H40zD6BTHq111rTEzA76Qa0pyAlSnu9FzQES1FS70a+ykomEegaJwmQPiNdA63G9BaYHWwAjZSnVQhkfVpr4cyAiDYgRVrwrrumDSegQJ2l12kh65Jrakxfh6WTsHzdbbGsYflErkrAgtblLbDW1DUgYhbMwkjLMAGjG62OZ2CJ/oL4P0KF3k/mC2szoS1KmhtrSGtIh0BSHGJAwSLQgyS+QAxAQ5AsRAqKguxFai6gBCxIrTBqiQiIcHryqhlQQ02IMoOKXqAnbzZAm4YaOie0orWnXqNqgRZtGrLQnnoNGoI9PBtH4a3BlQDPbPDiFe6eAT2y2yJKeJoBGamIFh02BLXQdGQK3tk55Rcmryk1yY90QAAAAABJRU5ErkJggg=="
item = gis.content.get(WEBMAP_ID)
wm = item.get_data()
with open(os.path.join(PASTA_BKP, "webmap_antes.json"), "w", encoding="utf-8") as f:
    json.dump(wm, f, ensure_ascii=False, indent=2)
ops = wm.get("operationalLayers", [])
par = next((ol for ol in ops if ol.get("id") == "paradas_onibus"), None)
if par is None:
    raise RuntimeError("Camada 'paradas_onibus' não encontrada no web map (rode a v10 antes).")
par.setdefault("layerDefinition", {})["drawingInfo"] = {"renderer": {"type": "simple", "symbol": {
    "type": "esriPMS", "imageData": SIMBOLO_PNG, "contentType": "image/png", "width": 17, "height": 17, "xoffset": 0, "yoffset": 0, "angle": 0}}}
# paradas por último = desenhadas por cima das linhas
ops.remove(par); ops.append(par)
wm["operationalLayers"] = ops
item.update(item_properties={"text": json.dumps(wm)})
log("  Símbolo aplicado e camada colocada por cima:", [o.get("id") for o in ops])
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Próximos passos =====
log("\n▶ CÉLULA 4 — Próximos passos: iniciando...")
log("  1) Reinicie o client (widget mudou): Ctrl+C na janela do client → S → cd C:\\ExB_1.19\\client → npm start")
log("  2) Construtor: Ctrl+F5 → mudança pequena → Salvar → Publicar")
log("  3) Teste: escolha a 195 → aparece a dica 💡; ligue Mostrar paradas → símbolos novos; passe o mouse numa parada → nome")
log("   Construtor (app 3): https://localhost:3001/builder/?id=3")
log("   Prévia local:       https://localhost:3001/experience/3/?locale=pt-br")
log("   Web map:            https://geo.apucarana.pr.gov.br/portal/apps/mapviewer/index.html?webmap=" + WEBMAP_ID)
log(f"  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 4 concluída com sucesso.")
